In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [2]:
import sys
sys.path.append("../scripts")

from data_loader import load_data

In [3]:
orders, order_items, products, refunds = load_data("../../data/raw")

In [4]:
# orders = pd.read_csv("../../data/raw/orders.csv")
# order_items = pd.read_csv("../../data/raw/order_items.csv")
# products = pd.read_csv("../../data/raw/products.csv")
# refunds = pd.read_csv("../../data/raw/order_item_refunds.csv")

In [5]:
orders["created_at"] = pd.to_datetime(
    orders["created_at"]
)

order_summary = (
    order_items
    .groupby("order_id")
    .agg(
        total_items=("order_item_id", "count"),
        revenue=("price_usd", "sum"),
        cogs=("cogs_usd", "sum")
    )
    .reset_index()
)

order_summary["profit"] = (
    order_summary["revenue"]
    - order_summary["cogs"]
)

orders_analysis = orders.merge(
    order_summary,
    on="order_id",
    how="left"
)

In [6]:
total_orders = orders_analysis["order_id"].nunique()

total_customers = orders_analysis["user_id"].nunique()

total_revenue = orders_analysis["revenue"].sum()

total_profit = orders_analysis["profit"].sum()

total_cogs = orders_analysis["cogs"].sum()

aov = total_revenue / total_orders

profit_margin = (
    total_profit / total_revenue
) * 100

final_kpis = pd.DataFrame({
    "Metric": [
        "Total Orders",
        "Total Customers",
        "Total Revenue",
        "Total COGS",
        "Total Profit",
        "Average Order Value",
        "Profit Margin"
    ],
    "Value": [
        total_orders,
        total_customers,
        total_revenue,
        total_cogs,
        total_profit,
        aov,
        profit_margin
    ]
})

display(final_kpis)

,Metric,Value
0,Total Orders,3.231300e+04
1,Total Customers,3.169600e+04
2,Total Revenue,1.938510e+06
3,Total COGS,7.223702e+05
4,Total Profit,1.216140e+06
5,Average Order Value,5.999164e+01
6,Profit Margin,6.273579e+01


In [7]:
product_sales = order_items.merge(
    products,
    on="product_id",
    how="left"
)

product_summary = (
    product_sales
    .groupby(
        ["product_id", "product_name"]
    )
    .agg(
        units_sold=("order_item_id", "count"),
        revenue=("price_usd", "sum"),
        cogs=("cogs_usd", "sum")
    )
    .reset_index()
)

product_summary["profit"] = (
    product_summary["revenue"]
    - product_summary["cogs"]
)

product_summary["profit_margin"] = (
    product_summary["profit"]
    / product_summary["revenue"]
) * 100

In [8]:
display(
    product_summary.sort_values(
        "revenue",
        ascending=False
    ).head(10)
)

,product_id,product_name,units_sold,revenue,cogs,profit,profit_margin
0,1,The Original Mr. Fuzzy,24226,1211057.74,472164.74,738893.0,61.012202
1,2,The Forever Love Bear,5796,347702.04,130352.04,217350.0,62.510418
2,3,The Birthday Sugar Panda,4985,229260.15,72232.65,157027.5,68.493151
3,4,The Hudson River Mini bear,5018,150489.82,47620.82,102869.0,68.356119


In [9]:
customer_summary = (
    orders_analysis
    .groupby("user_id")
    .agg(
        orders=("order_id", "nunique"),
        revenue=("revenue", "sum"),
        profit=("profit", "sum")
    )
    .reset_index()
)

customer_summary["customer_type"] = np.where(
    customer_summary["orders"] == 1,
    "One-time",
    "Repeat"
)

display(
    customer_summary.head()
)

,user_id,orders,revenue,profit,customer_type
0,13,1,49.99,30.5,One-time
1,20,1,49.99,30.5,One-time
2,59,1,49.99,30.5,One-time
3,104,1,49.99,30.5,One-time
4,147,1,49.99,30.5,One-time


In [10]:
refund_summary = (
    refunds
    .groupby("order_id")
    .agg(
        refund_amount=(
            "refund_amount_usd",
            "sum"
        )
    )
    .reset_index()
)

display(
    refund_summary.head()
)

,order_id,refund_amount
0,57,49.99
1,71,49.99
2,74,49.99
3,116,49.99
4,118,49.99


# Toylytics — Final Business Insights

## 1. Overall Business Performance

- Toylytics recorded **32,313 total orders** from **31,696 customers**.
- The business generated **$1,938,509.75 in total revenue**.
- Total COGS was **$722,370.25**, resulting in **$1,216,139.50 total profit**.
- The overall profit margin was **62.74%**.
- The average order value was **$59.99**.
- A total of **40,025 items** were sold.

---

## 2. Sales Performance

- **December 2014** recorded the highest monthly revenue of **$144,823.02**.
- **March 2012** recorded the lowest monthly revenue of **$2,999.40**.
- December 2014 also recorded the highest monthly profit of **$91,857.00**.
- The results show substantial variation in monthly sales performance across the analyzed period.

---

## 3. Product Performance

- **The Original Mr. Fuzzy** generated the highest revenue at **$1,211,057.74**.
- The Original Mr. Fuzzy also generated the highest total profit of **$738,893.00**.
- It was also the most sold product, with **24,226 units sold**.
- **The Birthday Sugar Panda** recorded the highest profit margin at **68.49%**.
- Product analysis shows that the product generating the highest sales volume and revenue was not necessarily the product with the highest profit margin.

---

## 4. Customer Behavior

- The dataset contains **31,105 one-time customers** and **591 repeat customers**.
- One-time customers generated **$1,864,153.31** in revenue.
- Repeat customers generated **$74,356.44** in revenue.
- The highest individual customer revenue recorded was **$251.94**.

---

## 5. Refund Analysis

- There were **1,731 refund records** in the dataset.
- Total refund amount was **$85,338.69**.
- **1,723 orders** had at least one refund record.
- The calculated refund rate was **5.33%** based on refunded orders relative to total orders.
- **The Original Mr. Fuzzy** had the highest refund amount at **$61,837.63**.

---

## 6. Key Business Insights

1. The business generated **$1.94 million in revenue** and **$1.22 million in profit**, with an overall profit margin of **62.74%**.

2. **The Original Mr. Fuzzy** was the dominant product by revenue, profit, and units sold, generating **$1.21 million in revenue** and selling **24,226 units**.

3. **The Birthday Sugar Panda** achieved the highest profit margin at **68.49%**, indicating that revenue leadership and margin leadership were different across products.

4. Customer activity was heavily concentrated among one-time customers: **31,105 one-time customers** compared with **591 repeat customers**.

5. Refunds totaled **$85,338.69**, and The Original Mr. Fuzzy accounted for **$61,837.63** of refund value, making it the product with the largest refund amount in the analysis.